In [1]:
import pandas as pd
import numpy as np
import re
import os
import torch
from datasets import Dataset
from transformers import AutoTokenizer

/home/ynt/.pyenv/versions/3.12.12/envs/data_eng/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.4.3)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


In [2]:
df = pd.read_csv('../data/cleaned/combined.csv')
print('Data file loaded')

Data file loaded


In [3]:
try:
    from Siamese import BurmeseConverter
except ImportError:
    from Siamese import BurmeseConverter

print("Loading configurations and downloading tokenizer...")
CSV_FILE_PATH = "../data/cleaned/combined.csv"
MODEL_NAME = "xlm-roberta-base"

converter = BurmeseConverter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def clean_and_syllable_map(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    
    def tokenize_myanmar_chunk(match):
        myanmar_text = match.group(0)
        try:
            result = converter.syllable_tokenization(1, myanmar_text)
            if result.startswith("With the virama mark:"):
                result = result.replace("With the virama mark:", "").strip()
            return " " + result + " "
        except Exception:
            return myanmar_text

    processed_text = re.sub(r'[\u1000-\u109F\uAA60-\uAA7F]+', tokenize_myanmar_chunk, text)
    processed_text = re.sub(r'\s+', ' ', processed_text).strip()
    
    return processed_text

if not os.path.exists(CSV_FILE_PATH):
    raise FileNotFoundError(f"Could not locate '{CSV_FILE_PATH}' in your current directory.")

print(f"Reading dataset from {CSV_FILE_PATH}...")
df = pd.read_csv(CSV_FILE_PATH)

print(f"Dataset loaded. Total rows: {len(df)}")
print("Running text syllable cleaning engine...")
df['cleaned_text'] = df['text'].astype(str).apply(clean_and_syllable_map)

print("Converting table matrix to Hugging Face mapping dataset...")
hf_dataset = Dataset.from_pandas(df[['cleaned_text', 'emotion_class']])

def batch_tokenize_step(examples):
    return tokenizer(
        examples["cleaned_text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

print("Executing SentencePiece subword tokenization across data batches...")
tokenized_dataset = hf_dataset.map(batch_tokenize_step, batched=True)

tokenized_dataset = tokenized_dataset.rename_column("emotion_class", "labels")
tokenized_dataset = tokenized_dataset.remove_columns(["cleaned_text"])
tokenized_dataset.set_format("torch")


Loading configurations and downloading tokenizer...


Reading dataset from ../data/cleaned/combined.csv...
Dataset loaded. Total rows: 3489
Running text syllable cleaning engine...
Converting table matrix to Hugging Face mapping dataset...
Executing SentencePiece subword tokenization across data batches...


Map:   0%|          | 0/3489 [00:00<?, ? examples/s]

In [4]:
df.columns

Index(['text', 'emotion_class', 'cleaned_text'], dtype='str')

In [5]:
df = df.drop(columns=['text'])

In [6]:
df.head(5)

,emotion_class,cleaned_text
0,negative,တ ကယ် ပဲ စိတ် ကုန် ဖို့ ကောင်း တယ်
1,negative,မင်း ကို လုံး ဝ ခွင့် မ လွှတ် နိုင် ဘူး
2,positive,ကမ် ဘာ ကြီး က ငါ့ အ ပေါ် ရက် စက် လွန်း တယ်
3,positive,တ ကယ် ပဲ လား ... ဆု တောင်း တွေ ပြည့် သွား တာ လား
4,positive,ဘယ် သူ့ ကို မှ မ တွေ့ ချင် တော့ ဘူး


In [7]:
df.columns

Index(['emotion_class', 'cleaned_text'], dtype='str')

In [ ]:
df['emotion_class'].value_counts()

In [47]:
df.to_csv('../data/processed/combined_processed.csv', index=False)
print("Saved")

Saved
